# Step 1: Raw Data — The Landing Zone

A data lake accepts data in its original form: whatever format the source system produces, without upfront modeling. This is the inverse of a data warehouse, where data is transformed and validated before it is loaded (schema-on-write).

This notebook generates a synthetic sales export — `date, region, product, quantity, revenue` — as a single CSV file. CSV is used deliberately as the starting point: it is row-based, uncompressed, and human-readable, representative of the format typically produced by a point-of-sale or ERP export when it first lands in a lake's raw zone.

In [ ]:
import os
from pathlib import Path

# Make sure we run from the repository root, regardless of the
# notebook's working directory
while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

## Generating the Raw CSV File

`scripts/generate_data.py` writes 300,000 rows in a series of chunks, producing a file of approximately 10–15 MB. This size is small enough to generate and query within seconds, and small enough to inspect manually in a spreadsheet, yet large enough to make differences in file format and query performance clearly measurable.

In [ ]:
!python scripts/generate_data.py --out lake/raw/sales.csv

## Look at the raw data

Open the file browser and find `lake/raw/sales.csv`. Note its size — we'll compare it against Parquet in the next notebook.

In [ ]:
import pandas as pd

csv_path = Path("lake/raw/sales.csv")
size_mb = csv_path.stat().st_size / (1024 * 1024)
print(f"{csv_path} is {size_mb:.1f} MB")

pd.read_csv(csv_path, nrows=10)

## Why CSV is a bad *permanent* home for this data

- **Row-based**: to read just the `revenue` column, every engine still has to scan every byte of every row.
- **No compression**: repeated values like region and product names are stored in full, every single time.
- **No embedded schema**: types (`quantity` is an int, `revenue` is a float) have to be *guessed* on every read.
- **No partitioning**: a query for a single month still means reading the entire file.

These four points are exactly what the next two notebooks fix — first with a better file format (Parquet), then with a query engine (DuckDB) that knows how to exploit it.